# Project 2: Maternal Health Risk Triage with Explainable AI
Predict risk level (low / mid / high) from vital signs, compare Random Forest and XGBoost, explain predictions with SHAP.


## Step 1: Setup

In [ ]:
!pip install -q shap kagglehub

from google.colab import drive
drive.mount('/content/drive')

import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score

save_dir = "/content/drive/MyDrive/Maternal_XAI_Project"
os.makedirs(save_dir, exist_ok=True)

## Step 2: Load the data
UCI Maternal Health Risk dataset (1,014 records).

In [ ]:
dataset_path = kagglehub.dataset_download("joebeachcapital/maternal-health-risk")
csv_files = [os.path.join(dataset_path, f) for f in os.listdir(dataset_path) if f.endswith(".csv")]

df = pd.read_csv(csv_files[0])
print(f"Loaded {len(df)} records. Columns: {list(df.columns)}")
print(df["RiskLevel"].value_counts())

## Step 3: Features, target, split
`stratify=y` keeps the class proportions the same in train and test, which matters on a small dataset.

In [ ]:
X = df.drop("RiskLevel", axis=1)
label_mapping = {"low risk": 0, "mid risk": 1, "high risk": 2}
y = df["RiskLevel"].str.lower().map(label_mapping)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Step 4: Train and compare Random Forest vs XGBoost
High-risk recall is the key triage metric: a missed high-risk patient is the worst error.

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(n_estimators=150, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5,
                             random_state=42, eval_metric="mlogloss")
}

results_summary = []
for name, clf in models.items():
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)

    acc = accuracy_score(y_test, preds) * 100
    macro_f1 = f1_score(y_test, preds, average="macro") * 100
    high_risk_recall = recall_score(y_test, preds, labels=[2], average="macro") * 100

    sample = X_test.iloc[[0]]
    start = time.perf_counter()
    for _ in range(500):
        clf.predict(sample)
    latency_ms = ((time.perf_counter() - start) / 500) * 1000

    results_summary.append((name, acc, high_risk_recall, macro_f1, latency_ms))
    print(f"{name:<14} acc {acc:.2f}% | high-risk recall {high_risk_recall:.2f}% | F1 {macro_f1:.2f}% | {latency_ms:.3f} ms")

## Step 5: Choose and save the model
In the original run, Random Forest scored higher on accuracy and high-risk recall, while XGBoost was faster and smaller. This cell saves XGBoost as the deployable model (speed and size). **Change `chosen` to `"Random Forest"` if you decide accuracy and recall matter more, and write your reason in the reflection.** SHAP below is run on the XGBoost model.

In [ ]:
chosen = "XGBoost"
best_model = models[chosen]

model_path = os.path.join(save_dir, "maternal_triage_model.joblib")
joblib.dump(best_model, model_path)
model_size_kb = os.path.getsize(model_path) / 1024
print(f"Saved {chosen}: {model_size_kb:.1f} KB")

## Step 6: SHAP explanation
Which vital signs drive the predictions?

In [ ]:
xgb_model = models["XGBoost"]
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(8, 5))
shap.summary_plot(shap_values, X_test, plot_type="bar",
                  class_names=["Low Risk", "Mid Risk", "High Risk"], show=False)
plt.title("Vital Signs Driving Maternal Risk Predictions (SHAP)")
plt.tight_layout()
shap_plot_path = os.path.join(save_dir, "shap_feature_importance.png")
plt.savefig(shap_plot_path, dpi=300)
plt.show()

## Step 7: Save the results summary

In [ ]:
lines = [
    "=======================================================",
    "MATERNAL HEALTH XAI TRIAGE - BENCHMARK RESULTS",
    "=======================================================",
    f"Dataset Size             : {len(df)} records (Test: {len(X_test)})",
]
for name, acc, hr_rec, f1, lat in results_summary:
    lines.append(f"{name:<14} -> Acc: {acc:.2f}% | High-Risk Recall: {hr_rec:.2f}% | F1: {f1:.2f}% | Latency: {lat:.3f} ms")
lines += [
    f"Saved model ({chosen}) size : {model_size_kb:.1f} KB",
    f"SHAP chart               : {shap_plot_path}",
    "=======================================================",
]
report_text = "\n".join(lines)
print(report_text)

with open(os.path.join(save_dir, "maternal_metrics.txt"), "w") as f:
    f.write(report_text)

## Limits to state honestly
Only 1,014 records from one population, a single random split with 203 test patients, so the numbers are noisy. This is a research prototype, not a clinical tool.